# Gemma Rescue Grid — Day-4 Production Synthesis (Kaggle 31B)
### Cloud-tier reasoning for the offline-first disaster response platform

**Kaggle Gemma 4 Good Hackathon · Tracks: Impact / Global Resilience + Special Tech / Cactus**

- **Code:** https://github.com/listyantidewi1/gemma-disaster-grid
- **Live dashboard:** https://nusasiaga.vercel.app
- **APK + writeup:** attached to the Kaggle submission

### What this notebook does

Three pre-curated scenarios of `EdgeTriageReport` objects (12 + 15 + 8
reports respectively, all originally generated by Gemma 4 E2B on a
responder's Android phone) are fed into Gemma 4 **31B** running on
Kaggle 2× T4 GPUs via Unsloth's 4-bit FastModel. For each scenario the
model emits a single `CommandCenterSynthesis` JSON object capturing
the consolidated operational picture: priority zones, recommended
actions, consolidated hazards, vulnerable groups, validity flags.

Each synthesis is saved to disk in two formats:

1. `outputs/synthesis_scenario_<id>.json` — schema-validated JSON, the
   canonical artefact.
2. `outputs/synthesis-scenario-<id>.ts` — TypeScript module ready to
   drop into the NusaSiaga dashboard at
   `nusasiaga/src/lib/synthesis-scenario-<id>.ts`.

Same Gemma 4 family, same JSON contract, top to bottom of the stack.

## 1. Environment setup

Kaggle gives us a 2× T4 (30 GB total VRAM) accelerator. With
Unsloth's 4-bit quantization, Gemma 4 31B uses ~17-20 GB so we have
comfortable headroom for the 16k-token context window each scenario
needs.

Versions are pinned to match the official Unsloth gemma-4-31b
starter notebook so reproductions don't drift.

In [ ]:
# ── Install dependencies FIRST ─────────────────────────────────────
%pip install -qqq \
    unsloth "unsloth_zoo>=2026.4.6" \
    "transformers==5.5.0" \
    "torch>=2.8.0" "triton>=3.4.0" \
    torchvision bitsandbytes torchcodec timm \
    pydantic
print("OK — dependencies installed.")

In [ ]:
# ── Clone the project repo ────────────────────────────────────────
# Idempotent: re-running this cell after we've chdir'd is safe.
import os, subprocess
REPO_URL = "https://github.com/listyantidewi1/gemma-disaster-grid.git"
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
REPO_DIR = os.path.join(BASE, "gemma-disaster-grid")

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
last = subprocess.run(
    ["git", "log", "-1", "--oneline"], capture_output=True, text=True
).stdout.strip()
print(f"CWD:    {os.getcwd()}")
print(f"Commit: {last}")

## 2. Load Gemma 4 31B and define the synthesise helper

`device_map="balanced"` spreads the model across both Kaggle T4s.
On Colab single-GPU this would need an A100; on Kaggle 2× T4 we
get the full 31B at 4-bit comfortably.

In [ ]:
# ── Model configuration ───────────────────────────────────────────
MODEL_NAME    = "unsloth/gemma-4-31B-it"   # the Kaggle submission target
DEVICE_MAP    = "balanced"                  # spread across both T4s
MAX_SEQ_LENGTH = 16384

print(f"Model:    {MODEL_NAME}")
print(f"Device:   {DEVICE_MAP}")
print(f"Seq len:  {MAX_SEQ_LENGTH}")

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,                # auto-detect bfloat16/float16
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    device_map=DEVICE_MAP,
)
print(f"\nLoaded {MODEL_NAME}")
print(f"GPU memory used: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
# ── synthesize(): one-shot Gemma 4 generation ─────────────────────
# Gemma 4 has no native system-role token; the Unsloth convention is
# to prepend the system text to the first user turn. We slice off the
# input prompt before decoding so our JSON extractor doesn't pick up
# the schema TEMPLATE from the prompt instead of the model's output.
from transformers import TextStreamer

def synthesize(system_prompt: str,
               user_content: str,
               max_new_tokens: int = 6000,
               stream: bool = True) -> str:
    full_user = f"{system_prompt}\n\n---\n\n{user_content}"
    messages = [{"role": "user", "content": [{"type": "text", "text": full_user}]}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        tokenize=True,
        return_dict=True,
    ).to("cuda")
    input_length = inputs["input_ids"].shape[1]
    kw = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )
    if stream:
        kw["streamer"] = TextStreamer(tokenizer, skip_prompt=True)
    outputs = model.generate(**kw)
    return tokenizer.decode(outputs[0, input_length:], skip_special_tokens=False)

print("synthesize() ready.")

## 3. Load the synthesis prompt and our schema utilities

In [ ]:
import sys, re, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

# Purge any cached grg modules so a fresh `git pull` is reflected.
for _mod in list(sys.modules):
    if _mod == "grg" or _mod.startswith("grg."):
        del sys.modules[_mod]

from grg import (
    EdgeTriageReport, CommandCenterSynthesis,
    parse_edge_report, parse_synthesis,
    extract_json_from_model_output,
    attempt_truncated_json_repair,
)

def load_system_prompt(md_path: str) -> str:
    """Extract the first triple-backtick fenced block from a .md prompt
    file. Lets us keep prompts human-readable as documentation while
    still being machine-extractable."""
    text = pathlib.Path(md_path).read_text(encoding="utf-8")
    m = re.search(r"```\s*\n(.*?)\n```", text, re.DOTALL)
    if not m:
        raise ValueError(f"No code fence in {md_path}")
    return m.group(1).strip()

SYNTHESIS_SYSTEM_PROMPT = load_system_prompt("prompts/cloud_synthesis_system.md")
print(f"System prompt: {len(SYNTHESIS_SYSTEM_PROMPT):,} chars "
      f"(~{len(SYNTHESIS_SYSTEM_PROMPT)//4} tokens)")

## 4. Scenario catalog

Three pre-curated scenarios. Each was hand-crafted to exercise a
different facet of the synthesis tier:

| ID | Disaster | Reports | Window | What it tests |
|---|---|---|---|---|
| A | Flood (rapid onset) | 12 | 90 min | Multiple priority zones, recurring electrical hazards, vulnerable evacuee |
| B | Shallow earthquake | 15 | 2 hours | Three sev-5 incidents, mass-casualty potential, low-confidence ambiguous report |
| C | Compound flood + electrical fires | 8 | 60 min | Reports disagree on primary disaster type — synthesis must produce a coherent compound classification |

Every report validates against the Pydantic `EdgeTriageReport` schema
(see `grg/smoke_test.py` for the test).

In [ ]:
        SCENARIOS = [
    {
        "id": "a",
        "label": "Rapid-onset flood",
        "json_path": "data/synthesis_scenarios/scenario_a_jakarta_flood.json",
        "n_reports": 12,
        "window_minutes": 90,
        "ts_module_name": "synthesis-scenario-a",
        "ts_export_name": "scenarioASynthesis"
    },
    {
        "id": "b",
        "label": "Shallow earthquake",
        "json_path": "data/synthesis_scenarios/scenario_b_cianjur_quake.json",
        "n_reports": 15,
        "window_minutes": 120,
        "ts_module_name": "synthesis-scenario-b",
        "ts_export_name": "scenarioBSynthesis"
    },
    {
        "id": "c",
        "label": "Compound flood + electrical fires",
        "json_path": "data/synthesis_scenarios/scenario_c_compound_flood_fire.json",
        "n_reports": 8,
        "window_minutes": 60,
        "ts_module_name": "synthesis-scenario-c",
        "ts_export_name": "scenarioCSynthesis"
    }
]
        from pathlib import Path
        OUTPUTS_DIR = Path("outputs")
        OUTPUTS_DIR.mkdir(exist_ok=True)
        print(f"Output dir: {OUTPUTS_DIR.resolve()}")
        for sc in SCENARIOS:
            assert Path(sc["json_path"]).exists(), f"missing {sc['json_path']}"
            print(f"  ✓ {sc['id'].upper()} {sc['label']}  ({sc['n_reports']} reports)")

## 5. Run synthesis for all three scenarios

Each scenario is processed independently:

1. Read the JSON file, parse and Pydantic-validate every report.
2. Build the user message: a brief framing line + the full reports
   array as a fenced JSON block.
3. Stream the model's generation. Gemma 4's `<|channel>thought` is
   captured then sliced off for the schema parse.
4. Extract the JSON object, validate against `CommandCenterSynthesis`.
   If validation fails, attempt the truncated-JSON repair pass.
5. Save the raw streaming output to a cache file so re-runs are cheap.

Wall-clock per scenario on Kaggle 2× T4:

- Scenario A (12 reports): ~6-8 min
- Scenario B (15 reports): ~8-12 min
- Scenario C (8 reports):  ~5-7 min

Total: ~25-30 min if all three regenerate from scratch.

In [ ]:
import json, time
from pathlib import Path

# Set to True for any scenario to force regeneration even if the
# cache file exists. Leave False to reuse cached output.
FORCE_REGEN = {"a": False, "b": False, "c": False}

results = {}

for sc in SCENARIOS:
    sid = sc["id"]
    print("\n" + "─" * 72)
    print(f"Scenario {sid.upper()}: {sc['label']}")
    print("─" * 72)

    scenario = json.loads(Path(sc["json_path"]).read_text(encoding="utf-8"))
    reports_raw = scenario["reports"]
    for r in reports_raw:
        _, err = parse_edge_report(r)
        if err:
            raise ValueError(f"  ✗ {r.get('report_id')}: {err}")
    print(f"  ✓ {len(reports_raw)} reports parsed and schema-validated")

    cache_path = Path(f"synthesis_cache_scenario_{sid}.txt")
    if cache_path.exists() and cache_path.stat().st_size > 1000 and not FORCE_REGEN[sid]:
        raw_output = cache_path.read_text(encoding="utf-8")
        print(f"  ✓ Loaded cache {cache_path.name} ({cache_path.stat().st_size:,} bytes)")
        wall = 0.0
    else:
        reports_json = json.dumps(reports_raw, indent=2)
        user_content = (
            f"Below are {len(reports_raw)} EdgeTriageReport objects "
            f"submitted over the past ~{sc['window_minutes']} minutes "
            f"from a developing {sc['label'].lower()} incident. Synthesize "
            f"them into a single CommandCenterSynthesis JSON object. "
            f"Include a <|channel>thought reasoning trace before the "
            f"final JSON.\n\n"
            f"```json\n{reports_json}\n```"
        )
        t0 = time.time()
        raw_output = synthesize(
            SYNTHESIS_SYSTEM_PROMPT,
            user_content,
            max_new_tokens=6000,
            stream=True,
        )
        wall = time.time() - t0
        cache_path.write_text(raw_output, encoding="utf-8")
        print(f"\n  ✓ Generated in {wall:.1f}s, cached to {cache_path.name}")

    results[sid] = {
        "raw_output": raw_output,
        "wall_clock_sec": wall,
        "n_reports": len(reports_raw),
        "scenario_meta": sc,
    }

print("\n" + "═" * 72)
print(f"All {len(results)} scenarios processed.")
print("═" * 72)

## 6. Extract and validate each synthesis as `CommandCenterSynthesis`

`extract_json_from_model_output` walks every balanced `{...}` block
in the raw output and returns the first one that round-trips through
`json.loads` AND validates against the Pydantic schema. If that fails
we try `attempt_truncated_json_repair` for the case where the model
hit `max_new_tokens` mid-write.

In [ ]:
import json
from pathlib import Path

validated = {}

for sid, res in results.items():
    raw = res["raw_output"]
    obj, err = extract_json_from_model_output(raw, parse_synthesis)
    if obj is None:
        print(f"  ⚠ {sid.upper()}: extract failed — trying repair pass")
        repaired = attempt_truncated_json_repair(raw)
        if repaired:
            obj, err = parse_synthesis(repaired)
    if obj is None:
        print(f"  ✗ {sid.upper()}: still no valid synthesis ({err}). "
              f"Inspect synthesis_cache_scenario_{sid}.txt manually.")
        continue

    json_path = Path(f"outputs/synthesis_scenario_{sid}.json")
    json_path.write_text(
        json.dumps(obj.model_dump(mode="json"), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"  ✓ {sid.upper()}: validated, wrote {json_path}")
    validated[sid] = obj

## 7. Emit drop-in TypeScript modules for the dashboard

Each synthesis becomes a TypeScript module exporting a typed constant.
Drop these into `nusasiaga/src/lib/` and `scenarios.ts` will pick them
up — that's how Scenario A's synthesis got rendered on the dashboard
from the Day-1 E4B run.

In [ ]:
        import json
        from pathlib import Path

        TS_HEADER_TEMPLATE = '''/**
 * AUTO-GENERATED by notebook/gemma_rescue_grid_kaggle.ipynb
 *
 * Scenario: {label}
 * Source:   {n_reports} EdgeTriageReports over a {window} minute window
 * Model:    {model_name} (4-bit via Unsloth)
 * Wall:     {wall_sec:.1f}s on Kaggle 2× T4
 * Run at:   {timestamp}
 *
 * Drop-in for nusasiaga/src/lib/. Imported by scenarios.ts and rendered
 * by FloodSynthesisPanel.
 */

import type {{ CommandCenterSynthesis }} from "./types";

export const {export_name}: CommandCenterSynthesis = '''

        from datetime import datetime, timezone
        now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

        for sid, obj in validated.items():
            sc = results[sid]["scenario_meta"]
            ts_path = Path(f"outputs/{sc['ts_module_name']}.ts")
            payload = json.dumps(obj.model_dump(mode="json"), indent=2, ensure_ascii=False)
            header = TS_HEADER_TEMPLATE.format(
                label=sc["label"],
                n_reports=sc["n_reports"],
                window=sc["window_minutes"],
                model_name=MODEL_NAME,
                wall_sec=results[sid]["wall_clock_sec"],
                timestamp=now,
                export_name=sc["ts_export_name"],
            )
            ts_path.write_text(header + payload + ";\n", encoding="utf-8")
            print(f"  ✓ {ts_path}  ({ts_path.stat().st_size:,} bytes)")

## 8. Summary

The cells below print a tabular summary of each synthesis: the
number of priority zones, the consolidated hazard count, the
severity distribution, and validity-flag counts. Useful for the
Kaggle writeup ("the 31B identified N priority zones across the
three scenarios, ranked M recommended actions").

In [ ]:
from pathlib import Path
try:
    import pandas as pd
    rows = []
    for sid, obj in validated.items():
        sc = results[sid]["scenario_meta"]
        ds = obj.model_dump(mode="json")
        rows.append({
            "id":          sid.upper(),
            "label":       sc["label"][:30],
            "reports":     ds["report_count"],
            "wall_sec":    f"{results[sid]['wall_clock_sec']:.0f}",
            "primary":     ds["primary_disaster_classification"]["type"],
            "p_zones":     len(ds["priority_zones"]),
            "hazards":     len(ds["consolidated_hazards"]),
            "actions":     len(ds["recommended_actions"]),
            "validity":    len(ds["report_validity_notes"]),
            "immediate":   ds["severity_distribution"].get("5", 0),
        })
    display(pd.DataFrame(rows))
except ImportError:
    for sid, obj in validated.items():
        ds = obj.model_dump(mode="json")
        print(f"{sid.upper()}: {len(ds['priority_zones'])} zones, "
              f"{len(ds['recommended_actions'])} actions, "
              f"{len(ds['consolidated_hazards'])} hazards")

In [ ]:
# ── Where to take the .ts files next ───────────────────────────────
print("Drop these into nusasiaga/src/lib/, then in scenarios.ts:")
print()
for sid, obj in validated.items():
    sc = results[sid]["scenario_meta"]
    print(f"  // scenario {sid.upper()}")
    print(f"  import {{ {sc['ts_export_name']} }} from \"./{sc['ts_module_name']}\";")
print()
print("Then flip the relevant SCENARIOS[<id>].synthesis to the imported const")
print("and synthesisStatus from 'pending' to 'generated'.")

---

## Next steps after this notebook finishes

1. **Download** every `outputs/synthesis-scenario-*.ts` from the
   Kaggle sidebar.
2. **Drop** them into `nusasiaga/src/lib/` (overwriting the
   Day-1 E4B Scenario A module if it's still there).
3. **Update** `nusasiaga/src/lib/scenarios.ts`:
    - Import the three new constants.
    - In each `SCENARIOS[id]` block, replace
      `synthesis: null` → the imported const, and
      `synthesisStatus: "pending"` → `"generated"`.
4. **Push** `nusasiaga` main; Vercel auto-rebuilds within ~60 s.
5. **Verify** at <https://nusasiaga.vercel.app>: Scenario B and C
   switch from the "synthesis pending" placeholder to the full
   command-center view.

That closes the full pipeline: Gemma 4 E2B on the phone produces
these reports → Gemma 4 31B in this notebook synthesises them →
the dashboard renders the operational picture. Every layer in
the same Gemma 4 family, every layer talking the same JSON
contract.